In [ ]:
import importlib
import processing_textblocks_helpers as pth
import os
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json
import fitz
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import spacy_stanza
import re
import unicodedata
from spacy.tokens import Token, Doc
from spacy.language import Language
from spacy.symbols import ORTH
import spacy
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex


In [ ]:
#importlib.reload(pth)

In [ ]:
source_path = "../data/emlap_sanitized_textblocks/"
len(os.listdir(source_path))

In [ ]:
filenames = os.listdir(source_path)
filename = filenames[10]
print(filename)

In [ ]:
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [ ]:
textblocks[1][:10]

In [7]:
doc, doc_margins = pth.process_with_source_tracking(textblocks[:], pth.nlp_latin)

In [8]:
work_id = filename[:6]
sent_dicts = pth.doc_to_sent_dicts(doc, work_id)
sent_dicts_margins = pth.doc_to_sent_dicts(doc_margins, work_id)

In [9]:
print(len(sent_dicts))
print(len(sent_dicts_margins))

25721
941


In [10]:
sent_dicts_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts, textblocks)
sent_dicts_margins_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts_margins, textblocks)

In [11]:
merged_sentences = pth.merge_main_and_margin_sentences(
    sent_dicts_with_coords,
    sent_dicts_margins_with_coords
)
len(merged_sentences)

26662

In [12]:
merged_sentences[:10]

[{'work_id': '100085',
  'sent_id': 0,
  'sent_text': 'Commentariorum',
  'tokens_data': [{'token_text': 'Commentariorum',
    'lemma': 'commentarius',
    'pos': 'NOUN',
    'ref': {'page': [1], 'textblock': [1], 'tag': '', 'blocktype': 'text'},
    'char_start': 0,
    'char_end': 14,
    'coordinates': [74.63999938964844,
     332.4238586425781,
     1787.7001953125,
     337.3438720703125]}]},
 {'work_id': '100085',
  'sent_id': 1,
  'sent_text': 'Alchymiae',
  'tokens_data': [{'token_text': 'Alchymiae',
    'lemma': 'alchymius',
    'pos': 'PROPN',
    'ref': {'page': [1], 'textblock': [2], 'tag': '', 'blocktype': 'title'},
    'char_start': 0,
    'char_end': 9,
    'coordinates': [612.239990234375,
     450.7439270019531,
     1403.43505859375,
     455.6639404296875]}]},
 {'work_id': '100085',
  'sent_id': 2,
  'sent_text': 'Andreae Libauii Med.',
  'tokens_data': [{'token_text': 'Andreae',
    'lemma': 'Andreas',
    'pos': 'ADJ',
    'ref': {'page': [1], 'textblock': [3], 'ta

In [13]:
merged_sentences[7]

{'work_id': '100085',
 'sent_id': 7,
 'sent_text': 'Prioreartis Libro Comprehensarum, Adiectis Fornacum Et Aliorum Uasorum Figuris, partim ex impressis antehac autoribus, partim aliunde acceptis, & ex latibulis officinarum productis.',
 'tokens_data': [{'token_text': 'Prioreartis',
   'lemma': 'prioreartus',
   'pos': 'ADP',
   'ref': {'page': [1], 'textblock': [7, 8], 'tag': '', 'blocktype': 'text'},
   'char_start': 0,
   'char_end': 11,
   'coordinates': [104.4000015258789,
    958.5840454101562,
    1768.1246337890625,
    963.5040283203125]},
  {'token_text': 'Libro',
   'lemma': 'Liber',
   'pos': 'NOUN',
   'ref': {'page': [1], 'textblock': [8], 'tag': '', 'blocktype': 'text'},
   'char_start': 12,
   'char_end': 17,
   'coordinates': [183.36000061035156,
    1033.4638671875,
    1669.21630859375,
    1038.3839111328125]},
  {'token_text': 'Comprehensarum',
   'lemma': 'comprehendo',
   'pos': 'NOUN',
   'ref': {'page': [1], 'textblock': [8], 'tag': '', 'blocktype': 'text'},
   

In [14]:
[s["sent_text"] for s in merged_sentences][:10]

['Commentariorum',
 'Alchymiae',
 'Andreae Libauii Med.',
 'D.',
 'Pars Prima,',
 'Sex Libris Declarata.',
 'Continens Explicationem Operationum Chymicarum',
 'Prioreartis Libro Comprehensarum, Adiectis Fornacum Et Aliorum Uasorum Figuris, partim ex impressis antehac autoribus, partim aliunde acceptis, & ex latibulis officinarum productis.',
 'Praemissa Est Defensio Alchemiae Et refutatio obiectionum ex Censura scholae Parisiensis, quae licet uideri nolit hanc Alchemiam, sed Quercetani damnasse, nimis tamen frigide de arte sentit, eaque proponit, quae in ludibrium & ignominiam artis simpliciter possunt conuerti, nec sonant aliter.',
 'Pag. 15.']

In [15]:
merged_tokens = pd.DataFrame(
    (lambda d, n=n: (d.update({"sent_n": n}) or d))(t)
    for n, token_list in enumerate([sent["tokens_data"] for sent in merged_sentences])
    for t in token_list
)
for attr in ["page", "textblock", "tag", "blocktype"]:
    merged_tokens[attr] = merged_tokens.apply(lambda row: row["ref"][attr], axis=1)
merged_tokens.drop(["ref", "coordinates"], axis=1, inplace=True)
merged_tokens.head(5)

,token_text,lemma,pos,char_start,char_end,sent_n,page,textblock,tag,blocktype
0,Commentariorum,commentarius,NOUN,0,14,0,[1],[1],,text
1,Alchymiae,alchymius,PROPN,0,9,1,[1],[2],,title
2,Andreae,Andreas,ADJ,0,7,2,[1],[3],,text
3,Libauii,Libauius,PROPN,8,15,2,[1],[3],,text
4,Med,ego,ADJ,16,19,2,[1],[3],,text


In [16]:
len(merged_tokens)

394698

In [17]:
merged_tokens["tag"].value_counts()

tag
      391234
GR      2825
G        340
S        297
I          2
Name: count, dtype: int64

In [18]:
merged_tokens[merged_tokens["tag"]=="S"]

,token_text,lemma,pos,char_start,char_end,sent_n,page,textblock,tag,blocktype
43288,◉,y,SYM,5,6,3113,[48],[0],S,margin
73799,◉,z,SYM,18,19,5672,[74],[31],S,text
73804,◉,z,SYM,9,10,5673,[74],[31],S,text
81471,◉,◉,SYM,35,36,6472,[82],[14],S,margin
81475,◉,◉,SYM,45,46,6472,[82],[15],S,margin
...,...,...,...,...,...,...,...,...,...,...
391941,◉,y,SYM,9,10,26456,[410],[48],S,text
391954,◉,y,SYM,0,1,26461,[410],[48],S,text
391972,◉,y,SYM,12,13,26467,[410],[48],S,text
391983,◉,y,SYM,0,1,26472,[410],[48],S,text


In [19]:
def process_textblocks(textblocks):
    filepath = os.path.join(source_path, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        textblocks = json.load(f)
    doc, doc_margins = pth.process_with_source_tracking(textblocks[:], pth.nlp_latin)
    work_id = filename[:6]
    sent_dicts = pth.doc_to_sent_dicts(doc, work_id)
    sent_dicts_margins = pth.doc_to_sent_dicts(doc_margins, work_id)
    sent_dicts_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts, textblocks)
    sent_dicts_margins_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts_margins, textblocks)
    merged_sentences = pth.merge_main_and_margin_sentences(
        sent_dicts_with_coords,
        sent_dicts_margins_with_coords
    )
    merged_sentences_uncoord = []
    for sent_data in merged_sentences:
        for token_data in sent_data["tokens_data"]:
            del token_data["coordinates"]
    return merged_sentences

In [ ]:
sents_data = process_textblocks(textblocks)

In [ ]:
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

In [27]:
for filename in os.listdir(source_path):
    id = filename[:6]
    outfile = id + ".json"
    if outfile not in os.listdir(target_path):
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        sent_dicts = process_textblocks(textblocks)
        outpath = os.path.join(target_path, outfile)
        with open(outpath, 'w', encoding='utf-8') as f:
            json.dump(sent_dicts, f, ensure_ascii=False, indent=2)